# CallGrader: fine-tune the LLM grader and serve it with vLLM (free Colab GPU)

Cross-validates ModernBERT, then trains Qwen2.5-0.5B-Instruct with LoRA to answer each call with a JSON verdict, evaluates it on the held-out
test calls with plain `transformers`, then serves the same merged model with **vLLM** and measures throughput.

1. **Runtime -> Change runtime type -> T4 GPU** (free).
2. Upload `callgrader-colab.zip` (made on the laptop by `scripts/colab_bundle.sh`) with the folder icon on the left.
3. **Runtime -> Run all.** About 1 hour. At the end, download `callgrader-results.zip` and unzip it into the repo.

In [ ]:
!unzip -oq callgrader-colab.zip -d /content/cg
%cd /content/cg
!pip -q install "vllm==0.11.*" "peft>=0.17" scikit-learn
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. ModernBERT: 5-fold cross-validation by scenario (about 30 min on a T4)
Same code as the laptop: `callgrader/cv.py`. Writes `results/cv.json` with the majority floor, the scikit-learn baseline and ModernBERT.

In [ ]:
!python -m callgrader.cv --encoder 2>&1 | grep -E "'fold'|^majority|^tfidf|^modernbert|Error"
import gc, torch; gc.collect(); torch.cuda.empty_cache()

## 1. Fine-tune with LoRA (PyTorch + PEFT)
Same code as on the laptop: `callgrader/llm.py`. Loss is on the JSON answer only; the adapter is merged into the base weights at the end.

In [ ]:
import json, time
from callgrader import llm
t0 = time.perf_counter()
history = llm.train(epochs=2)
print(history, f"{(time.perf_counter()-t0)/60:.1f} min")

## 2. Evaluate with plain transformers (one call at a time)

In [ ]:
res = {"model": llm.BASE + " + LoRA r16 (merged), greedy JSON", "history": history, "hardware": "Colab T4", **llm.evaluate()}
json.dump(res, open("results/llm.json", "w"), indent=2)
print(json.dumps(res["metrics"], indent=2), res["ms_per_call"], "ms/call", res["valid_json_rate"], "valid JSON")

## 3. Serve with vLLM (continuous batching, all test calls at once)
The prompt is built by the same `llm.prompt` function, so vLLM sees exactly what the model was trained on.

In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from callgrader.data import LABELS, read
from callgrader.metrics import report
tok = AutoTokenizer.from_pretrained("models/llm")
rows = read("test")
prompts = [llm.prompt(tok, r["text"]) for r in rows]
engine = LLM(model="models/llm", dtype="float16", max_model_len=2048, gpu_memory_utilization=0.6)
sp = SamplingParams(temperature=0, max_tokens=60)
engine.generate(prompts[:4], sp)                       # warm-up
t0 = time.perf_counter(); outs = engine.generate(prompts, sp); dt = time.perf_counter() - t0
preds = [llm.parse(o.outputs[0].text) for o in outs]
vres = {"engine": "vLLM 0.11 (fp16, continuous batching)", "hardware": "Colab T4", "calls": len(rows),
        "seconds": round(dt, 2), "calls_per_second": round(len(rows)/dt, 1), "ms_per_call": round(dt/len(rows)*1000, 1),
        "hf_ms_per_call": res["ms_per_call"], "speedup_vs_transformers": round(res["ms_per_call"]/(dt/len(rows)*1000), 1),
        "metrics": report(rows, preds, LABELS)}
json.dump(vres, open("results/vllm.json", "w"), indent=2)
print(json.dumps({k: v for k, v in vres.items() if k != "metrics"}, indent=2), "macro F1", vres["metrics"]["macro_f1"])

## 4. Download the results
`results/llm.json` and `results/vllm.json` feed the README table on the laptop (`python -m callgrader.compare`).

In [ ]:
!zip -q callgrader-results.zip results/cv.json results/llm.json results/vllm.json
from google.colab import files
files.download("callgrader-results.zip")